# PA4 - PySpark (Team 6, CineMatch)

## Task 1. Define and prepare the data

**Final project goal.** CineMatch recommends movies based on who worked on them. We need a clean table of IMDb credits (who did what on which title) that later steps can group, join, and rank.

**Analytical question 1.** Which crew roles show up in the top 10 billing spots of IMDb titles, and how high are they billed on average? (Same question as our PA3 benchmark.)

**Analytical question 2.** How does the mix of roles in the top 10 billing spots change across release eras?

**What one row is.** One credit: one person in one billing position on one title.

**Key.** `(title_id, billing_order)`. IMDb gives each credit on a title its own `ordering` number.

| column | type | units / meaning | source |
|---|---|---|---|
| `title_id` | string | IMDb title ID, `tt` + digits | `tconst` |
| `billing_order` | int | billing position, 1 = first credited | `ordering` |
| `person_id` | string | IMDb person ID, `nm` + digits | `nconst` |
| `role_category` | string | job type (actor, director, ...) | `category` |
| `is_top10` | boolean | derived: `billing_order <= 10` | new |
| `title_type` | string | movie, short, tvSeries, ... | `title.basics.titleType` (join) |
| `start_year` | int | release year, NULL if unknown | `title.basics.startYear` (join) |

`job` and `characters` are not needed for either question, so they are dropped.

### Setup
Install PySpark (Colab does not always have it) and start Spark in local mode. `local[2]` means two task threads on this one Colab VM, not two computers.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import csv, os

spark = (SparkSession.builder
         .appName("CS131-PA4-Team6")
         .master("local[2]")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

### Input files
The sample files are in the team repo (`pa4/data/sample/`). The repo is public, so Colab can clone it.

To run on the full data instead, change `PRINCIPALS_PATH` and `BASICS_PATH` to the full `title.principals.tsv` and `title.basics.tsv` (for example in a mounted Google Drive folder). Everything below stays the same.

In [ ]:
!rm -rf CineMatch-Netflix-Movies
!git clone -q https://github.com/sjsu-cs131-f26/CineMatch-Netflix-Movies.git

BASE = "CineMatch-Netflix-Movies/pa4"
PRINCIPALS_PATH = f"{BASE}/data/sample/principals_sample.tsv"
BASICS_PATH = f"{BASE}/data/sample/title_basics_lookup.tsv"

OUT_DIR = f"{BASE}/out"
PREPARED_PATH = f"{BASE}/work/prepared_credits.parquet"
os.makedirs(OUT_DIR, exist_ok=True)

# Every check is added to this list and saved to out/quality_checks.csv at the end.
checks = []
def record(check, stage, value, expected, note=""):
    result = "pass" if expected == "" or str(value) == str(expected) else "CHECK"
    checks.append({"check": check, "stage": stage, "value": value,
                   "expected": expected, "result": result, "note": note})
    print(f"{result:5} | {check}: {value}" + (f" (expected {expected})" if expected != "" else ""))

### 1. Read the input with an explicit schema
The file is tab-separated with one header row. We give the schema ourselves instead of letting Spark guess.

IMDb files do not use quoting, but some text fields contain `"` characters, so we turn quoting off with `quote = ""`. Otherwise a field that starts with `"` could be read across tabs and break the row.

In [ ]:
principals_schema = ("tconst STRING, ordering INT, nconst STRING, "
                     "category STRING, job STRING, characters STRING")

raw = (spark.read
       .option("header", True)
       .option("sep", "\t")
       .option("quote", "")
       .schema(principals_schema)
       .csv(PRINCIPALS_PATH))

raw.printSchema()
raw.orderBy("tconst", "ordering").show(5, truncate=False)

In [ ]:
# Line count straight from the file, to check that Spark read every data line.
with open(PRINCIPALS_PATH) as f:
    file_lines = sum(1 for _ in f)

raw_rows = raw.count()
record("raw data lines in file (minus header)", "read", file_lines - 1, "")
record("rows read by Spark", "read", raw_rows, file_lines - 1,
       "should equal the file's data lines; a difference means rows were split or merged while parsing")

### 2. Select the fields we need and set names

In [ ]:
credits = raw.select(
    F.col("tconst").alias("title_id"),
    F.col("ordering").alias("billing_order"),
    F.col("nconst").alias("person_id"),
    F.col("category").alias("role_category"))

credits.printSchema()

### 3. Quality condition 1: missing or invalid required values
IMDb writes missing values as the text `\N`. A credit cannot be used if:
- the title or person ID is missing or does not look like an IMDb ID (`tt...` / `nm...`),
- the billing order is missing (it was not a whole number) or below 1,
- the role category is missing or `\N`.

**Treatment:** exclude those rows. They cannot be placed in a billing position or a role group. This is the same rule PA3's `prepare_data.sh` used, so Analysis 1 matches PA3.

A row can break more than one rule, so the per-rule counts can add up to more than the number of excluded rows. We report each rule and also the number of distinct rows excluded.

In [ ]:
bad_title    = F.col("title_id").isNull() | ~F.col("title_id").like("tt%")
bad_person   = F.col("person_id").isNull() | ~F.col("person_id").like("nm%")
bad_order    = F.col("billing_order").isNull() | (F.col("billing_order") < 1)
bad_category = F.col("role_category").isNull() | (F.col("role_category") == "\\N")
any_bad = bad_title | bad_person | bad_order | bad_category

record("rows with missing/invalid title_id", "quality 1", credits.filter(bad_title).count(), "")
record("rows with missing/invalid person_id", "quality 1", credits.filter(bad_person).count(), "")
record("rows with missing or < 1 billing_order", "quality 1", credits.filter(bad_order).count(), "")
record("rows with missing role_category", "quality 1", credits.filter(bad_category).count(), "")

excluded = credits.filter(any_bad).count()
record("distinct rows excluded (any rule)", "quality 1", excluded, "",
       "rules can overlap, so this is not the sum of the four counts above")

valid = credits.filter(~any_bad)
valid_rows = valid.count()
record("rows kept after quality 1", "quality 1", valid_rows, raw_rows - excluded)

### 4. Quality condition 2: inconsistent role labels
Most IMDb role labels use one word, but some use underscores (`archive_footage`, `archive_sound`, `production_designer`). PA3 changed these three to hyphens with `sed`, so we do the same here. Then Analysis 1 uses exactly the PA3 labels.

**Treatment:** rename the three labels. No rows are removed. `casting_director` was not in the PA3 rule, so we leave it as is to match the PA3 baseline and note it as a known inconsistency.

In [ ]:
label_fix = {"archive_footage": "archive-footage",
             "archive_sound": "archive-sound",
             "production_designer": "production-designer"}

relabeled = valid.filter(F.col("role_category").isin(list(label_fix.keys()))).count()
record("rows with an underscore label renamed", "quality 2", relabeled, "")

cleaned = valid.replace(label_fix, subset=["role_category"])
record("rows after quality 2", "quality 2", cleaned.count(), valid_rows, "relabeling should not add or remove rows")

### 5. Key uniqueness and domain values
Each `(title_id, billing_order)` should appear once. We only report repeated keys. We do **not** remove them, because two rows with the same key could be different people, and we have no rule for choosing one.

In [ ]:
repeated_keys = (cleaned.groupBy("title_id", "billing_order").count()
                 .filter(F.col("count") > 1))
record("repeated (title_id, billing_order) keys", "key", repeated_keys.count(), 0,
       "reported only, not removed")

role_values = (cleaned.groupBy("role_category").count()
               .orderBy(F.col("count").desc(), F.col("role_category").asc()))
role_values.show(truncate=False)
record("distinct role_category values", "domain", role_values.count(), "",
       "list: " + "; ".join(r["role_category"] for r in role_values.collect()))

record("min billing_order", "domain", cleaned.agg(F.min("billing_order")).collect()[0][0], "", "must be >= 1")
record("max billing_order", "domain", cleaned.agg(F.max("billing_order")).collect()[0][0], "")
record("distinct titles", "domain", cleaned.select("title_id").distinct().count(), "")

### 6. Derived column
`is_top10` marks credits in the top 10 billing spots. Both questions use it: Analysis 1 counts only these credits, and Analysis 2 compares them across eras.

In [ ]:
prepared_base = cleaned.withColumn("is_top10", F.col("billing_order") <= 10)

top10_rows = prepared_base.filter(F.col("is_top10")).count()
record("credits with is_top10 = true", "derived", top10_rows, "")
record("credits with is_top10 = false", "derived",
       prepared_base.filter(~F.col("is_top10")).count(), cleaned.count() - top10_rows)

### 7. Join with the title lookup (release year)
Question 2 needs each title's release year, which is in `title.basics`. One lookup row is one title, with `title_id` as the key.

**Relationship:** many credits to one title. We use a **left join**, so every credit is kept even if its title has no lookup row. Before joining we check that the lookup has no repeated `title_id`; a repeat would duplicate credits. `\N` in `startYear` fails the INT type and becomes NULL, which means the year is unknown.

In [ ]:
basics_schema = ("tconst STRING, titleType STRING, primaryTitle STRING, originalTitle STRING, "
                 "isAdult STRING, startYear INT, endYear STRING, runtimeMinutes STRING, genres STRING")

titles = (spark.read
          .option("header", True)
          .option("sep", "\t")
          .option("quote", "")
          .schema(basics_schema)
          .csv(BASICS_PATH)
          .select(F.col("tconst").alias("title_id"),
                  F.col("titleType").alias("title_type"),
                  F.col("startYear").alias("start_year")))

titles.printSchema()

record("lookup rows", "join", titles.count(), "")
record("repeated title_id in lookup", "join",
       titles.groupBy("title_id").count().filter(F.col("count") > 1).count(), 0,
       "must be 0 or the join would duplicate credits")

In [ ]:
before_join = prepared_base.count()
prepared = prepared_base.join(titles, on="title_id", how="left")
after_join = prepared.count()

record("rows before join", "join", before_join, "")
record("rows after left join", "join", after_join, before_join, "a left join with a unique lookup keeps the row count")
record("credits with no matching title", "join",
       prepared.filter(F.col("title_type").isNull()).count(), "",
       "title_type is never NULL in the lookup, so NULL here means no match")
record("credits with unknown start_year", "join",
       prepared.filter(F.col("start_year").isNull()).count(), "",
       "includes unmatched titles and titles whose startYear is \\N")

prepared.orderBy("title_id", "billing_order").show(5)

### 8. Save as Parquet, read it back, and verify
The Parquet "file" is a folder of part files. We compare the row count, the column names and types, and two project totals: the number of top-10 credits, and the sum of billing positions.

In [ ]:
prepared.write.mode("overwrite").parquet(PREPARED_PATH)
restored = spark.read.parquet(PREPARED_PATH)

restored.printSchema()

record("Parquet read-back rows", "parquet", restored.count(), after_join)
record("Parquet read-back schema", "parquet", str(restored.dtypes), str(prepared.dtypes),
       "column names and types in order")
record("Parquet read-back is_top10 count", "parquet",
       restored.filter(F.col("is_top10")).count(), top10_rows)
record("Parquet read-back sum of billing_order", "parquet",
       restored.agg(F.sum("billing_order")).collect()[0][0],
       prepared.agg(F.sum("billing_order")).collect()[0][0])

### 9. Save the checks
This writes one real CSV file with a header (not a Spark part-file folder).

In [ ]:
checks_path = f"{OUT_DIR}/quality_checks.csv"
with open(checks_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["check", "stage", "value", "expected", "result", "note"])
    writer.writeheader()
    writer.writerows(checks)

print("Saved", len(checks), "checks to", checks_path)
print("Checks needing a look:", [c["check"] for c in checks if c["result"] != "pass"])

### Keep the outputs
Colab runtime files disappear after a reset. Download `out/quality_checks.csv`, or copy the outputs to the team's Google Drive folder:

```python
from google.colab import drive
drive.mount("/content/drive")
!cp -r CineMatch-Netflix-Movies/pa4/out "/content/drive/MyDrive/<team folder>/pa4_out"
!cp -r CineMatch-Netflix-Movies/pa4/work/prepared_credits.parquet "/content/drive/MyDrive/<team folder>/"
```

The sampling and split checks for `quality_checks.csv` come from Task 3 and are added there.